In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType


In [0]:
connection_string = dbutils.secrets.get(
    scope="azure-secrets",
    key="eventhub-connection-string"
)

eventhub_name = "orders-eventhub"

# El conector Kafka de Event Hub usa el puerto 9093 y este formato de autenticación SASL
eh_namespace = "evhyanquiel.servicebus.windows.net"
bootstrap_servers = f"{eh_namespace}:9093"

sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{connection_string}";'
)

kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": eventhub_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": sasl_config,
    "startingOffsets": "earliest",  # lee desde el principio, no se pierde lo que el producer ya envió
}

In [0]:
event_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("item_id", IntegerType()),
    StructField("item_name", StringType()),
    StructField("price", DoubleType()),
    StructField("event_timestamp", StringType()),
])

In [0]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

In [0]:
orders_stream = (
    raw_stream
    .selectExpr("CAST(value AS STRING) as json_value")
    .select(from_json(col("json_value"), event_schema).alias("data"))
    .select("data.*")
)

orders_enriched = orders_stream.withColumn("_ingest_timestamp", current_timestamp())


In [0]:
query = (
    orders_enriched.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/dbr_dev/yanquiel_dabs/landing/_checkpoints/eventhub/orders")
    .trigger(availableNow=True)
    .toTable("dbr_dev.yanquiel_dabs.brz_event_orders")
)


In [0]:
query.awaitTermination()
print("Ingestion complete into brz_event_orders")